<a href="https://colab.research.google.com/github/Swarit1212/ai_practice/blob/main/nanoGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-17 16:28:51--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2026-09-17 16:28:51 (157 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [ ]:
with open('input.txt', 'r',encoding= 'utf-8') as f:
  text=f.read();

In [ ]:
len(text)

1115394

In [ ]:
text[:1000]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [ ]:
vocab=sorted(list(set(text)))
vocab_size=len(vocab)
print(vocab)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [ ]:
stoi ={ch:i for i,ch in enumerate(vocab)}
itos={i:ch for i,ch in enumerate(vocab)}

def encode(s):
  return [stoi[i] for i in s]
def decode(l):
  return ''.join([itos[i] for i in l])

In [ ]:
data=torch.tensor(encode(text),dtype=torch.long).to(device)

In [ ]:
n=int(0.9 * len(data))
train_data=data[:n]
test_data = data[n:]

In [ ]:
block_size=256
batch_size=64

torch.manual_seed(1212)
def get_batch(split):
  data=train_data if split=='train' else test_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size] for i in ix]).to(device)
  y=torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)
  return x,y

xb,yb=get_batch('train')
print(xb)
print(yb)

tensor([[ 1, 39, 57,  ..., 43, 47, 56],
        [43, 42,  0,  ...,  1, 46, 39],
        [43, 41, 58,  ..., 59, 11,  1],
        ...,
        [ 1, 41, 53,  ..., 47, 51, 43],
        [50, 50,  1,  ..., 46, 43,  1],
        [46,  1, 16,  ..., 60, 39, 47]], device='cuda:0')
tensor([[39, 57,  1,  ..., 47, 56,  1],
        [42,  0, 35,  ..., 46, 39, 60],
        [41, 58,  1,  ..., 11,  1, 39],
        ...,
        [41, 53, 51,  ..., 51, 43,  1],
        [50,  1, 46,  ..., 43,  1, 45],
        [ 1, 16, 59,  ..., 39, 47, 50]], device='cuda:0')


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

n_embd =384

class Head(nn.Module):
  def __init__(self,head_size):
    super().__init__()
    self.key = nn.Linear(n_embd,head_size,bias=False)
    self.query = nn.Linear(n_embd,head_size,bias=False)
    self.value = nn.Linear(n_embd,head_size,bias=False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    self.dropout = nn.Dropout(0.2)
  def forward(self,x):
    B,T,C = x.shape
    k = self.key(x)
    q = self.query(x)
    wei = q @ k.transpose(-2,-1)*C**(-0.5)
    wei = wei.masked_fill(self.tril[:T,:T]==0 ,float('-inf'))
    wei = F.softmax(wei,dim=-1)
    wei = self.dropout(wei)
    v=self.value(x)
    out = wei @ v
    return out


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__ (self,num_heads,head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
    self.proj = nn.Linear(n_embd,n_embd)
    self.dropout = nn.Dropout(0.2)
  def forward(self,x):
    return self.dropout(self.proj(torch.cat([h(x) for h in self.heads],dim=-1)))


In [ ]:
class feedForward(nn.Module):
  def __init__(self,n_embd):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embd,4*n_embd),
        nn.ReLU(),
        nn.Linear(4*n_embd,n_embd),
        nn.Dropout(0.2)
    )
  def forward(self,x):
      return self.net(x)

In [ ]:
class Blocks(nn.Module):
  def __init__(self,n_embd,n_head):
    super().__init__()
    head_size = n_embd // n_head
    self.sa = MultiHeadAttention(n_head,head_size)
    self.ffwd = feedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)
  def forward(self,x):
    x = x+self.sa(self.ln1(x))
    x=x+self.ffwd(self.ln2(x))
    return x

In [ ]:
n_head=6
n_layer=6
class BigramLanguageModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
    self.position_embedding_table = nn.Embedding(block_size,n_embd)
    self.blocks = nn.Sequential(*[Blocks(n_embd,n_head) for i in range(n_layer)])
    self.ln_f = nn.LayerNorm(n_embd)
    self.lm_head = nn.Linear(n_embd,vocab_size)

  def forward(self,idx,targets=None):
    B,T = idx.shape
    tok_emb =self.token_embedding_table(idx)
    pos_emb=self.position_embedding_table(torch.arange(T,device=device))
    x=tok_emb + pos_emb
    x=self.blocks(x)
    x=self.ln_f(x)
    logits = self.lm_head(x)
    if targets is None:
      loss=None
    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits,targets)
    return logits,loss
  def generate (self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      # crop idx to the last block_size tokens
      idx_cond = idx[:, -block_size:]
      # get the predictions
      logits,loss=self(idx_cond)
      # focus only on the last time step
      logits=logits[:,-1,:] # becomes (B, C)
      # apply softmax to get probabilities
      probs=F.softmax(logits,dim=-1) # (B, C)
      # sample from the distribution
      idx_next=torch.multinomial(probs,num_samples=1) # (B, 1)
      # append sampled index to the running sequence
      idx=torch.cat((idx,idx_next),dim=1) # (B, T+1)
    return idx

xb,yb=get_batch('train')

m= BigramLanguageModel().to(device)
logits,loss=m(xb,yb)
print(logits.shape)
print(loss)


torch.Size([16384, 65])
tensor(4.3201, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
eval_interval=500
eval_iters=200
max_iters=5000
@torch.no_grad()
def estimate_loss():
  out={}
  m.eval()
  for split in ['train','val']:
    losses=torch.zeros(eval_iters)
    for k in range(eval_iters):
      xb,yb=get_batch(split)
      logits,loss=m(xb,yb)
      losses[k]=loss.item()
    out[split]=losses.mean()
  m.train()
  return out

In [ ]:
optimizer = torch.optim.AdamW(m.parameters(),lr=3e-4)

In [ ]:
batch_size=32
for i in range(max_iters):
  if(i%eval_interval==0):
    losses=estimate_loss()
    print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
  xb,yb=get_batch('train')
  logits,loss = m(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()
print(loss.item())

step 0: train loss 4.3143, val loss 4.3201
step 500: train loss 2.1226, val loss 2.1766
step 1000: train loss 1.7206, val loss 1.8681
step 1500: train loss 1.5437, val loss 1.7350
step 2000: train loss 1.4451, val loss 1.6573
step 2500: train loss 1.3717, val loss 1.5872
step 3000: train loss 1.3185, val loss 1.5505
step 3500: train loss 1.2794, val loss 1.5255
step 4000: train loss 1.2486, val loss 1.5056
step 4500: train loss 1.2180, val loss 1.5019
1.3058040142059326


In [ ]:
idx=torch.zeros((1,1) ,dtype=torch.long, device=device)
print(decode(m.generate(idx,max_new_tokens=300)[0].tolist()))


And hear me, that yet from all my queen,
And twere that last, I'ld be newn them, as sight hategraynce
Than Lury thou spurr'd: he had my loving their keep:
I warrant, tell me what the bold
That with this floots world.
Maring me, my lords, thy tribe one that's; if thence!
This marring him, now, thou w


self attention

### Steps before pushing your notebook to GitHub:

1.  **Clear all outputs**: This ensures your notebook runs fresh and doesn't contain any potentially outdated or environment-specific outputs.
    *   Go to `Runtime` > `Restart and run all`.
    *   Then go to `Edit` > `Clear all outputs`.

2.  **Remove sensitive information**: Make sure you don't have any API keys, personal credentials, or other sensitive data hardcoded in your notebook. If you used Colab secrets, that's generally fine, but double-check that no raw keys are present.

3.  **Ensure all necessary files are accessible**: If your notebook relies on external data files (like `input.txt` in this case), make sure they are either included in your GitHub repository, publicly accessible via a URL (like your `wget` command), or clearly documented on how to obtain them.

4.  **Add comments and explanations**: Make sure your code is well-commented and that markdown cells clearly explain the purpose of each section, the methodology used, and the insights gained. This makes your notebook understandable to others (and your future self).

5.  **Install dependencies**: List all the Python libraries your notebook uses in a `requirements.txt` file or explicitly install them at the beginning of the notebook using `!pip install` commands. This ensures others can easily replicate your environment.

6.  **Test reproducibility**: Run the entire notebook from scratch (after clearing outputs) on a fresh environment (e.g., another Colab session or locally) to confirm everything works as expected.

7.  **Review commit message**: Write a clear and concise commit message that describes the changes you've made or the purpose of the notebook.

8.  **Add a `README.md`**: Consider adding a `README.md` file to your repository that explains what the notebook does, how to run it, what libraries are needed, and any notable results or conclusions.


In [ ]:
torch.save(m.state_dict(),'model.pth')